# F5 語音與影片工作室
## 雙引擎與字幕開關的程式逐行講義

讀完本講義，你應能從一個控制項的 .value 出發，指出它存在哪個 JSON 欄位、哪個 Python 程序會讀它，以及它最後改變音檔、字幕還是影片畫面。

純語音教學版有兩條路：第 1 至 7 格將文字譯成多語、人工校閱後交給 Chatterbox；第 8A 至 8C 格則在目前的 Colab 環境直接呼叫 F5，獨立輸出一個 WAV。後者不會自動沿用前一條路的多語校閱表。

影片版先在主環境翻譯分鏡。校閱後按生成，worker 才依 tts_engine 選擇主環境或 .f5venv。兩個引擎的音訊都交給同一段影片程式：每語言一支 MP4、一份 SRT 和一份時間軸；開啟燒錄會另保留乾淨影片。

每節只讀一段原始邏輯。L 編號是該 Colab 表單格或內嵌引擎字串的實際行號，由 1 起算；整段縮排為閱讀方便平移。原碼節錄供對讀，不應把半個 if 或命令清單複製成獨立程式。

全書 34 節。配套 Notebook 將原碼與逐行註解放在閱讀格，另附 10 個可執行實驗。從頭執行不需 GPU、不安裝套件、不下載權重；真實模型生成請回到附件的 Colab 工作室。

比對結果：單獨上傳的影片 Notebook 與 ZIP／notebooks 下的 SubtitleToggle 逐位元組一致；純語音以同資料夾下的 F5_TTS_Teaching 為準。ZIP／original_notebooks 是對照版本，沒有用舊版覆蓋新版。

## 閱讀順序

**語音工作室：一段文字產生 WAV**

01　從校閱欄位建立多語語音請求

02　把 F5 安裝在當前的 Python 環境

03　從候選錄音選出參考檔

04　決定純語音版實際要念的文字

05　將參考音和新旁白交給 F5 並寫檔

**影片工作室：從畫面送出請求**

06　讓 Perth 的初始化錯誤提早出現

07　為影片版 F5 準備獨立直譯器

08　將引擎及字幕選擇轉成有型別的值

09　取場景快照並固定播放順序

10　校閱時保存編輯器及快照簽章

11　設定 F5 專用的三個控制項

12　生成前組裝雙層場景請求

13　依動作與引擎選擇 worker 程序

**素材前處理：從瀏覽器到工作引擎**

14　接收瀏覽器的音訊資料塊

15　檢查參考 WAV 的數值與長度

16　對過長文字持續取出前綴

**生成引擎：翻譯、驗證與波形**

17　取得目標語言的翻譯 token

18　驗證引擎與字幕旗標

19　轉型後檢查 F5 參數

20　用物件封裝 F5 模型與共用狀態

21　把共同 generate 呼叫轉接為 F5 infer

22　在迴圈外選定模型

23　分開字幕文字與模型要念的文字

24　把多段波形寫成一個場景 WAV

**媒體輸出：每一格影像與每一條字幕**

25　統一圖片方向與透明背景

26　將旁白秒數向上對齊影格

27　將靜態圖和旁白編成一個片段

28　從同一個 cursor 產生 SRT 與時間軸

29　先合成乾淨影片並寫出外掛字幕

30　燒錄成功時保留兩種影片

31　燒錄失敗時還原檔案並回報

32　區分可以繼續的失敗與 GPU 記憶體不足

33　只封裝成功語言的成果

34　以檔案大小決定 Notebook 預覽方式

## 01　從校閱欄位建立多語語音請求

**來源：** 純語音教學版第 6 格　L9–L12

這是純語音版第 6 格的 Chatterbox 路徑。字典中的 rows 是一維清單，每列是一種語言；影片版則多一層 scenes，把每張圖連到多語文字。按鈕名稱相似，worker 的輸入契約不同。

```python
if not editors:
    raise ValueError('請先執行第 5 格並校對文字。')
latest_result = worker('synthesize', {'audio':reference, 'generation':generation,
    'rows':[{'language':language, 'text':editor.value} for language, editor in editors]})
```

**逐行讀法**

- **L009**　空的 editors 表示沒有可供合成的校閱列。
- **L010**　拋出 ValueError，避免用尚未校閱的資料產生音檔。
- **L011**　將錄音路徑、生成參數放入請求，啟動 synthesize 工作。
- **L012**　清單推導式逐一拆開 (language, editor)，直到此刻才讀取 editor.value。

**追蹤一次執行：** 若兩個編輯框分別含「你好」與 Hello，rows 的長度是 2；改第二個框只改第二種語音。這裡不會進入後面的獨立 F5 示範。

## 02　把 F5 安裝在當前的 Python 環境

**來源：** 純語音教學版第 8A 格　L5–L7

純語音版的第 8A 格沒有鎖定上述三個套件的版本。它會改變當前 kernel 環境；影片版的第 1A 格則將 F5 安裝到 .f5venv。讀到 subprocess.run 時，要先看命令清單的第一項。

```python
print("正在背景安裝 F5-TTS 核心與依賴套件，首次執行約需 1-2 分鐘...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "f5-tts", "torchaudio", "cached-path"], check=True)
print("F5-TTS 安裝完成！請執行下一格進行語音合成。")
```

**逐行讀法**

- **L005**　這行只顯示提示，不保證安裝所需時間。
- **L006**　以 sys.executable 指定目前 kernel 的 Python；-m pip 選擇該 Python 的安裝工具，check=True 將非零退出碼變成例外。
- **L007**　僅在上一行成功後顯示完成訊息。

**追蹤一次執行：** 兩個 Notebook 都寫 import F5TTS，卻可能載入不同環境的套件。檢查安裝位置比重複按執行更有用。

## 03　從候選錄音選出參考檔

**來源：** 純語音教學版第 8B 格　L32–L40

8B 先看已存在的全域 reference；找不到才掃描 /content 和 /content/voice_lab，再套用這段選檔邏輯。都沒有候選檔才提示上傳。檔名比對不會辨認說話者或音質。

```python
keywords = ['record', 'reference', '下載']
selected = None
for f in possible_files:
    if any(k in f.name.lower() for k in keywords):
        selected = f
        break
if not selected:
    selected = possible_files[0]
ref_audio_path = str(selected)
```

**逐行讀法**

- **L032**　列出優先尋找的檔名片段。
- **L033**　以 None 表示尚未選到。
- **L034**　依候選清單原本的順序走訪。
- **L035**　any 對生成器逐項判斷；檔名轉小寫後包含任一關鍵字即通過。
- **L036**　記住找到的 Path。
- **L037**　break 只結束尋找檔案的這個迴圈。
- **L038**　找不到關鍵字時走備援分支。
- **L039**　取清單第一項；這裡外層已確認清單非空。
- **L040**　將 Path 轉成後續推論需要的字串路徑。

**追蹤一次執行：** 候選清單是 [a.wav, record_2.wav] 時優先選第二個；若只有 a.wav，就選它。開始合成前應看清畫面顯示的檔名。

## 04　決定純語音版實際要念的文字

**來源：** 純語音教學版第 8B 格　L60–L63

strip() 回傳新的字串；空字串在條件中視為 False。這裡以實際賦值順序為準：手動表單會覆蓋控制項。後面的 ref_text 是參考錄音中已經說過的內容，兩者扮演不同角色。

```python
# 最終決定合成的文本
target_text = target_text_input.strip() if target_text_input.strip() else text_from_widget
if not target_text:
    target_text = "同學晚安，今天我們要來體驗語音AI的技術"
```

**逐行讀法**

- **L060**　註解標出最後的文字選擇步驟。
- **L061**　A if 條件 else B：手動表單去空白後有內容就優先採用，否則取第 4 格控制項文字。
- **L062**　兩個來源都空白時，才進入預設分支。
- **L063**　指定內建示範句給 target_text。

**追蹤一次執行：** 第 4 格為「歡迎來到圖書館」，手動表單為「請看借閱規則」，最後念的是後者。

**動手試驗**　把手動欄位改成只有空白，觀察是哪個來源成為旁白。

預期：`請看指示`

In [ ]:
def choose_text(form, editor):
    target = form.strip() if form.strip() else editor
    return target or '同學晚安，今天我們要來體驗語音AI的技術'
assert choose_text(' ', '歡迎來到圖書館') == '歡迎來到圖書館'
print(choose_text('請看指示', '歡迎來到圖書館'))

## 05　將參考音和新旁白交給 F5 並寫檔

**來源：** 純語音教學版第 8B 格　L75–L82

純語音 8B 直接建立 F5TTS()，此格允許 ref_text 留空並交給套件處理；影片版在前端和 worker 都明確拒絕空逐字稿。不要把純語音示例的寬鬆輸入套進影片工作室。檔名中的 taiwanese 也不是口音保證。

```python
# 生成音訊
output_wav = "/content/f5_taiwanese_output.wav"
wav, sr, spect = f5tts.infer(
    ref_file=ref_audio_path,
    ref_text=ref_text,
    gen_text=target_text,
    file_wave=output_wav
)
```

**逐行讀法**

- **L075**　接下來開始推論及寫檔。
- **L076**　指定固定目的檔路徑；重跑仍指向同一個檔名。
- **L077**　infer 傳回三項，依序拆包成波形、取樣率和第三項頻譜結果。
- **L078**　ref_file 是參考錄音的位置。
- **L079**　ref_text 描述參考錄音實際念的內容。
- **L080**　gen_text 才是新旁白，來自上一段選定的 target_text。
- **L081**　file_wave 指定寫出的 WAV。
- **L082**　括號結束這一次 infer 呼叫。

**追蹤一次執行：** 成功後 8C 先以 Path.is_file() 檢查 /content/f5_taiwanese_output.wav 再下載。這條路線沒有圖片、字幕或影片打包。

## 06　讓 Perth 的初始化錯誤提早出現

**來源：** 影片版第 1 格　L19–L25

相鄰的 Python 字串常值會自動接成一段命令。原第 1 格先裝 ffmpeg、fonts-noto-cjk 和主環境的固定套件；此探測與 1A 的 F5 匯入測試是兩件事。

```python
subprocess.run([PYTHON, '-c',
    'import torch; import pkg_resources; import perth; '
    'from perth.perth_net.perth_net_implicit.perth_watermarker import PerthImplicitWatermarker; '
    'assert callable(perth.PerthImplicitWatermarker), "Perth initialization unavailable"; '
    'watermarker = PerthImplicitWatermarker(); '
    'from chatterbox.mtl_tts import ChatterboxMultilingualTTS; '
    'assert torch.cuda.is_available(), "GPU unavailable"; print("GPU、模型套件與 Perth 浮水印初始化檢查完成")'], check=True)
```

**逐行讀法**

- **L019**　以主虛擬環境 PYTHON 另啟程序，-c 表示執行接下來的字串。
- **L020**　匯入 torch、套件資源及 perth；字串尾端的分號分隔 Python 敘述。
- **L021**　明確從深層模組匯入真實浮水印類別。
- **L022**　assert 先確認公開的 PerthImplicitWatermarker 可以呼叫，而非 None。
- **L023**　實際建立實例，檢查的不只是名稱能否匯入。
- **L024**　確認該環境可匯入 Chatterbox 類別。
- **L025**　檢查 CUDA，印出狀態；check=True 讓測試失敗反映到呼叫端。

**追蹤一次執行：** 若在這裡無法建立 watermarker，調整 temperature 不會解決模型初始化問題。成功也只代表小型探測通過，沒有生成測試語音。

## 07　為影片版 F5 準備獨立直譯器

**來源：** 影片版第 1A 格　L4–L10

清單的 + 是串接兩份參數，不是拼接 shell 字串。即使環境已存在，後面的兩次安裝仍會執行。隨後只測試匯入與 GPU；模型權重要到生成時才載入。

```python
F5_PYTHON = str(BASE / '.f5venv/bin/python')
if not Path(F5_PYTHON).exists():
    subprocess.run(uv + ['venv','--python','3.11',str(BASE / '.f5venv')],check=True)
subprocess.run(uv + ['pip','install','--python',F5_PYTHON,
    'torch==2.8.0','torchaudio==2.8.0','--index-url','https://download.pytorch.org/whl/cu128'],check=True)
subprocess.run(uv + ['pip','install','--python',F5_PYTHON,
    'f5-tts==1.1.22','Pillow==11.3.0','opencc-python-reimplemented==0.1.7','soundfile==0.13.1'],check=True)
```

**逐行讀法**

- **L004**　利用 Path 的 / 運算子接出 .f5venv/bin/python，再轉成命令可用的字串。
- **L005**　先檢查該直譯器路徑是否存在。
- **L006**　若缺少便用前一格提供的 uv 建立 Python 3.11 虛擬環境。
- **L007**　將 pip 安裝命令指向 F5_PYTHON，這個清單接續下一行。
- **L008**　鎖定 torch 與 torchaudio 2.8.0，指定 cu128 套件來源。
- **L009**　再次指定相同的 F5 環境來安裝應用依賴。
- **L010**　鎖定 f5-tts 1.1.22 等套件；安裝失敗時停止。

**追蹤一次執行：** 只用 Chatterbox 時可不執行 1A；選 F5 做影片則必須讓 worker 找得到這個直譯器。翻譯仍用主環境。

## 08　將引擎及字幕選擇轉成有型別的值

**來源：** 影片版第 4 格　L16–L19

命名參數控制元件行為。style 和 layout 只改呈現方式，不會改變傳入模型的設定。勾選代表把字畫進影片，不是在播放器裡開關文字軌。

```python
tts_engine = widgets.Dropdown(options=[('Chatterbox Multilingual', 'chatterbox'), ('F5-TTS v1 Base（中英）', 'f5')], value='chatterbox', description='語音引擎', style={'description_width':'100px'}, layout=widgets.Layout(width='95%'))
ref_transcript = widgets.Textarea(value='', description='錄音逐字稿', placeholder='F5必填：參考錄音中實際說的話，不是每張圖的新旁白', layout=widgets.Layout(width='98%',height='90px'), style={'description_width':'100px'})
burn_subtitles = widgets.Checkbox(value=False, description='將字幕顯示在影片畫面（燒錄）', indent=False, layout=widgets.Layout(width='98%'))
display(source, targets, video_size, tts_engine, ref_transcript, burn_subtitles)
```

**逐行讀法**

- **L016**　Dropdown 的選項是（顯示名稱，傳回值）；預設 .value 是 chatterbox。
- **L017**　Textarea 接收參考逐字稿；placeholder 是提示，不是已填好的文字。
- **L018**　Checkbox 預設 False，.value 是布林值。
- **L019**　在同一介面顯示語言、畫面尺寸及這三個控制項。

**追蹤一次執行：** 預設目標含日本語。改選 F5 後應將目標改為本影片版接受的繁體中文或 English；未改會在生成前被擋下。

## 09　取場景快照並固定播放順序

**來源：** 影片版第 4 格　L49–L54

排序後的新清單不改動畫面原來的控制項。此快照只涵蓋 scenes、source、targets；更換字幕開關本身不會使譯稿過期。

```python
orders = [scene['order'].value for scene in scene_widgets]
if sorted(orders) != list(range(1, len(scene_widgets) + 1)):
    raise ValueError('播放順序不可重複，請填入 1 到圖片總數。')
scenes = [{'image':s['image'], 'text':s['text'].value.strip()}
    for s in sorted(scene_widgets, key=lambda s:s['order'].value)]
return {'scenes':scenes, 'source':source.value, 'targets':list(targets.value)}
```

**逐行讀法**

- **L049**　清單推導式從每個控制項讀出 order.value。
- **L050**　比較排序後的編號與 1 到 N；range 的終點不包含在內。
- **L051**　缺號、跳號或重複都會中止。
- **L052**　每個場景只保留路徑和去除首尾空白的文字。
- **L053**　用 lambda 讀取 order.value 當排序鍵，再依序建立上述字典。
- **L054**　回傳普通 dict/list，而非無法直接序列化的 widget。

**追蹤一次執行：** 上傳順序 A、B，但播放順序填 2、1，送入翻譯的第一筆是 B；填 1、1 則根本不會送出。

**動手試驗**　用 ZIP 的真實旁白列練習，反向讀入後按順序欄位還原。

預期：`歡迎來到圖書館。請先看清楚各區的指示。`

In [ ]:
rows = [{'順序': '1', '圖片': 'scene_01.png', '繁體中文原文': '歡迎來到圖書館。請先看清楚各區的指示。', '英文校對示例': 'Welcome to the library. Please read the signs for each section.'}, {'順序': '2', '圖片': 'scene_02.png', '繁體中文原文': '借書前請先確認借閱規則。借閱期限請以館方公告為準。', '英文校對示例': 'Please check the borrowing rules before borrowing a book. Follow the loan period announced by the library.'}]
ordered = sorted(reversed(rows), key=lambda row: int(row["順序"]))
assert [r["圖片"] for r in ordered] == ["scene_01.png", "scene_02.png"]
print(ordered[0]["繁體中文原文"])

## 10　校閱時保存編輯器及快照簽章

**來源：** 影片版第 5 格　L13–L20

編輯器物件要留到按生成時才取 .value，否則使用者後來的校正會消失。sort_keys=True 固定字典順序，但不會改變場景清單的順序。此 signature 是比對字串，不是安全簽章。

```python
    text_widgets = {}
    for language in snapshot['targets']:
        editor = widgets.Textarea(value=scene['texts'][language], description=language,
            layout=widgets.Layout(width='98%', height='100px'), style={'description_width':'100px'})
        text_widgets[language] = editor
        display(editor)
    reviewed_scenes.append({'image':scene['image'], 'editors':text_widgets})
review_signature = json.dumps(snapshot, ensure_ascii=False, sort_keys=True)
```

**逐行讀法**

- **L013**　每個場景重新建立一份字典，防止不同圖共用編輯器。
- **L014**　依本次快照的目標語言逐一處理。
- **L015**　以該語言的翻譯作為文字框的初值。
- **L016**　設定編輯區尺寸與標籤寬度，完成建構。
- **L017**　以語言名稱為鍵記住控制項本身。
- **L018**　顯示此語言的編輯框。
- **L019**　把圖片路徑和該圖的所有編輯器收進 reviewed_scenes。
- **L020**　把原始快照序列化，鍵排序固定，供生成前比對。

**追蹤一次執行：** 修改校閱框內的英文是正常流程；若更改原始中文、場景順序或目標語言，需要回到翻譯格更新快照。

## 11　設定 F5 專用的三個控制項

**來源：** 影片版第 5A 格　L26–L30

字典的值是控制項物件，不是數字；送出時才取 .value。Chatterbox 另用 temperature、exaggeration、cfg_weight、top_p。兩種 cfg 的名稱及介面不同，不能只改拼字就當成同一參數。

```python
f5_controls = {
 'speed': widgets.FloatSlider(value=1.0,min=0.6,max=1.5,step=0.05,description='F5 speed'),
 'cfg_strength': widgets.FloatSlider(value=2.0,min=0.0,max=5.0,step=0.1,description='F5 cfg'),
 'nfe_step': widgets.Dropdown(options=[16,32,64],value=32,description='F5 steps')
}
```

**逐行讀法**

- **L026**　另建 f5_controls，不和 Chatterbox 的四個欄位混用。
- **L027**　speed 是速度倍率，預設 1.0，範圍 0.6–1.5。
- **L028**　cfg_strength 是引導強度，預設 2.0，範圍 0–5。
- **L029**　nfe_step 只提供 16、32、64，預設 32。
- **L030**　結束控制項字典。

**追蹤一次執行：** 利用 ZIP 中參數比較紀錄.csv：C、D 除了 speed 還改了字幕開關。要比較旁白速度，請固定文字、參考音和其他條件，另記錄音訊長度。

## 12　生成前組裝雙層場景請求

**來源：** 影片版第 6 格　L17–L21

這段有兩層推導式：外層產生場景列表，內層以語言作鍵。兩套參數可以同時存在於 JSON；後端只驗證及啟用本次引擎的參數。前面的守門條件要求有效參考錄音、有效快照，以及 F5 的非空逐字稿。

```python
request = {'tts_engine':tts_engine.value, 'ref_text':ref_transcript.value.strip(),
    'f5_generation':{k:v.value for k,v in f5_controls.items()},
    'burn_subtitles':burn_subtitles.value, 'audio':reference, 'generation':generation, 'targets':list(targets.value), 'size':video_size.value,
    'scenes':[{'image':s['image'], 'texts':{lang:w.value for lang,w in s['editors'].items()}}
        for s in reviewed_scenes]}
```

**逐行讀法**

- **L017**　以引擎值和去空白的參考逐字稿開始組裝 request。
- **L018**　字典推導式將每個 F5 控制項轉成數值。
- **L019**　同時保留字幕旗標、錄音路徑、Chatterbox 參數、目標語言和影片尺寸。
- **L020**　每張圖內再建立 texts 字典，讀取各語言校閱框目前的值。
- **L021**　外層清單推導式走訪所有已校閱場景，閉合 request。

**追蹤一次執行：** 兩張圖、兩種語言形成 2 個 scenes、共 4 個 texts 值。ref_text 只有一份，它對應共同參考錄音，不應填入其中一張圖的旁白。

## 13　依動作與引擎選擇 worker 程序

**來源：** 影片版第 2 格　L24–L29

request 是寫入 JSON 後交給子程序的資料，不是直接共享記憶體。Popen 可以邊執行邊讀日誌。回報 NoneType 或 ModuleNotFoundError 時，應查被選中的直譯器，而非只在 Notebook kernel 測試 import。

```python
runtime_python = str(BASE / '.f5venv/bin/python') if action == 'synthesize' and request.get('tts_engine') == 'f5' else PYTHON
if not Path(runtime_python).is_file():
    raise RuntimeError('F5環境尚未安裝，請先執行第1A格。')
process = subprocess.Popen([runtime_python, '-u', str(BASE / 'video_engine.py'),
    action, str(req), str(res)], stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
```

**逐行讀法**

- **L024**　兩個條件同時成立才用 .f5venv：正在合成，且請求的引擎是 f5；其餘均用 PYTHON。
- **L025**　檢查選出的直譯器是否為檔案。
- **L026**　未找到時停止；這段錯誤文字統一提示執行 1A。
- **L027**　用選出的直譯器啟動 video_engine.py；-u 讓日誌不緩衝。
- **L028**　將動作與請求、回應檔路徑當成獨立命令參數；以 PIPE 接標準輸出。
- **L029**　將標準錯誤合併至標準輸出，以 UTF-8 讀文字，遇解碼錯誤使用替代字元。

**追蹤一次執行：** translate + f5 → 主環境；synthesize + chatterbox → 主環境；synthesize + f5 → .f5venv。

**動手試驗**　將原條件抽成純函式，檢查翻譯是否也會進 F5 環境。

預期：`['.venv', '.f5venv', '.venv']`

In [ ]:
def route(action, request):
    return '.f5venv' if action == 'synthesize' and request.get('tts_engine') == 'f5' else '.venv'
actual = [route(a, {'tts_engine': e}) for a,e in [
    ('translate','f5'), ('synthesize','f5'), ('synthesize','chatterbox')]]
assert actual == ['.venv','.f5venv','.venv']
print(actual)

## 14　接收瀏覽器的音訊資料塊

**來源：** 影片版第 2 格內嵌 RECORDER_JS　L20–L26

JavaScript 的錄音事件發生在瀏覽器，不在 Colab GPU。停止後將 chunks 組成 Blob，FileReader 讀成 data URL，再由 Python 還原成檔案。此處的 chunks 裝音訊位元組，和 Python 切文字的 chunks 函式沒有關係。

```javascript
stream = await navigator.mediaDevices.getUserMedia({audio:true});
const mime = ['audio/webm;codecs=opus', 'audio/mp4', 'audio/webm']
  .find(type => MediaRecorder.isTypeSupported(type));
recorder = new MediaRecorder(stream, mime ? {mimeType:mime} : undefined);
const chunks = [];
recorder.ondataavailable = event => { if (event.data.size) chunks.push(event.data); };
recorder.onerror = () => reject(new Error('錄音失敗，請改用上傳錄音檔。'));
```

**逐行讀法**

- **L020**　await 等待瀏覽器取得音訊串流；audio:true 不代表指定了某款 USB 麥克風。
- **L021**　以優先順序列出希望使用的容器和編碼。
- **L022**　find 用箭頭函式逐項詢問支援度，取首個可用格式。
- **L023**　三元運算式決定是否明確指定 mimeType，再建立錄音器。
- **L024**　宣告本次錄音的資料塊陣列。
- **L025**　指定事件處理函式：收到非空資料才 push 進陣列。
- **L026**　錄音器出錯時用 reject 結束 Promise。

**追蹤一次執行：** 有波形不代表選對播放裝置。若上傳檔能播放、錄音預覽無聲，先分開測試瀏覽器錄音來源、接收檔及播放器輸出，不要先改模型取樣參數。

## 15　檢查參考 WAV 的數值與長度

**來源：** 影片版第 2 格內嵌 voice_engine.py　L104–L111

前段 FFmpeg 最多解碼前 31 秒，轉成每秒 24000 樣本的單聲道 WAV。單聲道時 len(audio)/sr 為秒數。isfinite(...).all() 將逐樣本結果彙整為單一布林值。

```python
audio, sr = sf.read(output)
if len(audio) < sr * 3:
    raise ValueError("錄音至少需要 3 秒；建議 6–15 秒清晰人聲。")
if not np.isfinite(audio).all() or np.max(np.abs(audio)) < 0.002:
    raise ValueError("錄音沒有可用音量，請重新錄音。")
if len(audio) > sr * 30:
    raise ValueError("錄音請控制在 30 秒以內。")
return output
```

**逐行讀法**

- **L104**　soundfile 回傳音訊陣列和取樣率。
- **L105**　比較樣本數與三秒對應的樣本數。
- **L106**　太短時拒絕，並提供建議錄製時長。
- **L107**　確認每一點都是有限值，並以絕對值峰值檢查音量下限。
- **L108**　數值異常或峰值太低就拒絕。
- **L109**　比較樣本數是否超過三十秒。
- **L110**　超長時拒絕，不默默裁成合格樣本。
- **L111**　回傳已轉換的檔案路徑。

**追蹤一次執行：** 120000 個樣本 ÷ 24000 Hz = 5 秒，符合時長。若全為零，仍會被峰值檢查擋下；若只是很大的雜音，卻可能通過。

## 16　對過長文字持續取出前綴

**來源：** 影片版第 2 格內嵌 voice_engine.py　L33–L41

原函式先依標點後方及換行分句，預設上限為 180 個字元。這裡僅限制字元數；翻譯還要在 tokenizer 後重新檢查 token 數。

```python
    while len(part) > limit:
        cut = part.rfind(" ", 0, limit + 1)
        if cut < limit // 2:
            cut = limit
        result.append(part[:cut].strip())
        part = part[cut:].strip()
    if part:
        result.append(part)
return result
```

**逐行讀法**

- **L033**　while 在剩餘文字仍超限時重複執行。
- **L034**　rfind 在上限附近往前找最後一個空白；無空白回傳 -1。
- **L035**　若切點太前面，連同找不到的情形一起處理。
- **L036**　改以 limit 作切點。
- **L037**　切片 [:cut] 不含 cut 位置，去除首尾空白後保存。
- **L038**　將剩餘後綴重新指定給 part，保證下一輪變短。
- **L039**　迴圈結束後檢查是否還有非空尾段。
- **L040**　將尾段加入結果。
- **L041**　回傳保持順序的清單。

**追蹤一次執行：** 若 limit=4，輸入 abcdefghij 得到 abcd、efgh、ij。若拿掉更新 part 的一行，迴圈將反覆處理同一段。

**動手試驗**　直接執行附件中完整的 chunks 函式，將字元上限縮到 4。

預期：`['abcd', 'efgh', 'ij']`

In [ ]:
import re
def chunks(text, limit=180):
    """Preserve all non-whitespace characters; avoid silent model truncation."""
    parts = re.split(r"(?<=[。！？.!?；;])\s*|\n+", text.strip())
    result = []
    for part in parts:
        part = part.strip()
        while len(part) > limit:
            cut = part.rfind(" ", 0, limit + 1)
            if cut < limit // 2:
                cut = limit
            result.append(part[:cut].strip())
            part = part[cut:].strip()
        if part:
            result.append(part)
    return result
parts = chunks("abcdefghij", 4)
assert parts == ["abcd","efgh","ij"]
print(parts)

## 17　取得目標語言的翻譯 token

**來源：** 影片版第 2 格內嵌 video_engine.py　L60–L67

外層遇到來源與目標語言相同時直接複製文字、continue；第一次需要跨語翻譯才載入 NLLB。解碼後的字串交給校閱框，並不直接呼叫 TTS。

```python
for part in chunks(scene['text']):
    inputs = tokenizer(part, return_tensors='pt')
    if inputs.input_ids.shape[-1] > 500:
        raise ValueError("句子太長，請分段後再試。")
    with torch.inference_mode():
        tokens = model.generate(**inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(LANGUAGES[target][1]),
            max_new_tokens=512, num_beams=3)
```

**逐行讀法**

- **L060**　將一張圖的原文拆段。
- **L061**　tokenizer 把文字轉為 PyTorch 張量。
- **L062**　讀取 input_ids 最後一維；此處比較的是 token 數而非字數。
- **L063**　超過 500 就拒絕此段。
- **L064**　在不建立梯度追蹤的推論區塊中生成。
- **L065**　**inputs 將編碼結果展開成具名參數。
- **L066**　取 NLLB 的目標語言代碼並轉成 token ID，指定輸出的語言。
- **L067**　最多生成 512 個新 token，束搜尋保留 3 條候選。

**追蹤一次執行：** 同一個繁中標籤，NLLB 查 LANGUAGES[target][1]；稍後 TTS 路徑查 [0]。不能把 zho_Hant 和 zh 任意互換。

## 18　驗證引擎與字幕旗標

**來源：** 影片版第 2 格內嵌 video_engine.py　L183–L194

UI 驗證方便操作，worker 驗證則保護直接從 JSON 呼叫的路徑。檢查逐字稿非空不等於確認文字和錄音吻合；這需要實際校聽。

```python
def validate_engine_options(request):
    engine_name = request.get('tts_engine', 'chatterbox')
    if engine_name not in {'chatterbox', 'f5'}:
        raise ValueError('語音引擎無效。')
    if not isinstance(request.get('burn_subtitles', False), bool):
        raise ValueError('字幕開關必須為布林值。')
    if engine_name == 'f5':
        if any(x not in {'繁體中文', 'English'} for x in request['targets']):
            raise ValueError('本教學版F5-TTS僅開放繁體中文與English；其他語言請切回Chatterbox。')
        if not isinstance(request.get('ref_text'), str) or not request['ref_text'].strip():
            raise ValueError('F5-TTS請填寫與參考錄音一致的逐字稿，不是欲生成的旁白。')
    return engine_name
```

**逐行讀法**

- **L183**　宣告後端的請求驗證函式。
- **L184**　缺少 tts_engine 時採用 chatterbox。
- **L185**　只接受兩個列舉字串；集合成員運算不比較顯示名稱。
- **L186**　未知引擎立刻中止。
- **L187**　字幕欄位預設 False；已提供時必須是真正 bool。
- **L188**　拒絕字串或數字等含混輸入。
- **L189**　只有 F5 繼續檢查語言和逐字稿。
- **L190**　any 找出任一不在允許集合內的目標語言。
- **L191**　遇到不支援語言即中止整個請求。
- **L192**　先確認 ref_text 是 str，再呼叫 strip；or 的短路避免對 None 呼叫方法。
- **L193**　空白或非字串逐字稿不得進入 F5。
- **L194**　回傳驗證好的引擎名稱。

**追蹤一次執行：** 傳入字串 'False' 會被拒絕，不會當作關閉字幕。F5 的中英文限制是這份應用的規則，不能推成所有 F5 權重的能力界線。

**動手試驗**　執行附件原函式；檢查合法 F5、空逐字稿、未開放語言與假布林。

預期：`三種錯誤請求均被拒絕`

In [ ]:
def validate_engine_options(request):
    engine_name = request.get('tts_engine', 'chatterbox')
    if engine_name not in {'chatterbox', 'f5'}:
        raise ValueError('語音引擎無效。')
    if not isinstance(request.get('burn_subtitles', False), bool):
        raise ValueError('字幕開關必須為布林值。')
    if engine_name == 'f5':
        if any(x not in {'繁體中文', 'English'} for x in request['targets']):
            raise ValueError('本教學版F5-TTS僅開放繁體中文與English；其他語言請切回Chatterbox。')
        if not isinstance(request.get('ref_text'), str) or not request['ref_text'].strip():
            raise ValueError('F5-TTS請填寫與參考錄音一致的逐字稿，不是欲生成的旁白。')
    return engine_name
valid = dict(tts_engine='f5', targets=['繁體中文'], ref_text='大家好', burn_subtitles=False)
assert validate_engine_options(valid) == 'f5'
for change in [dict(ref_text=' '), dict(targets=['日本語']), dict(burn_subtitles='False')]:
    try: validate_engine_options({**valid, **change})
    except ValueError: pass
    else: raise AssertionError(change)
print('三種錯誤請求均被拒絕')

## 19　轉型後檢查 F5 參數

**來源：** 影片版第 2 格內嵌 video_engine.py　L197–L208

要分辨『原始型別』與『轉型後的值』。這版先 int 再檢查集合，因此 32.9 可截成 32；float('1.1') 可以通過。它沒有像 Chatterbox 分支一樣逐一拒絕所有未知鍵。

```python
def f5_settings(request):
    values = request.get('f5_generation', {})
    speed = float(values.get('speed', 1.0))
    cfg = float(values.get('cfg_strength', 2.0))
    steps = int(values.get('nfe_step', 32))
    if not (math.isfinite(speed) and 0.6 <= speed <= 1.5):
        raise ValueError('F5 speed需在0.6至1.5。')
    if not (math.isfinite(cfg) and 0 <= cfg <= 5):
        raise ValueError('F5 cfg_strength需在0至5。')
    if steps not in {16, 32, 64}:
        raise ValueError('F5 nfe_step需選16、32或64。')
    return {'speed': speed, 'cfg_strength': cfg, 'nfe_step': steps}
```

**逐行讀法**

- **L197**　宣告從 request 讀 F5 設定的函式。
- **L198**　缺少整組設定時取空字典。
- **L199**　取速度值轉 float，預設 1.0。
- **L200**　取引導強度轉 float，預設 2.0。
- **L201**　取步數轉 int，預設 32。
- **L202**　要求 speed 有限且在閉區間內。
- **L203**　速度不合法則停止。
- **L204**　引導強度也要有限且介於 0 與 5。
- **L205**　不符合就回報引導強度錯誤。
- **L206**　轉型後的步數必須在三個整數中。
- **L207**　步數不合法則拒絕。
- **L208**　回傳可用 ** 展開給 infer 的字典。

**追蹤一次執行：** speed=1.6 會拋 ValueError；nfe_step=48 也不合法。若呼叫者傳 f5_generation=None，會在 .get 階段失敗，不是自動套用預設。

**動手試驗**　用實際函式觀察轉型先於白名單檢查的結果。

預期：`{'speed': 1.1, 'cfg_strength': 2.0, 'nfe_step': 32}`

In [ ]:
import math
def f5_settings(request):
    values = request.get('f5_generation', {})
    speed = float(values.get('speed', 1.0))
    cfg = float(values.get('cfg_strength', 2.0))
    steps = int(values.get('nfe_step', 32))
    if not (math.isfinite(speed) and 0.6 <= speed <= 1.5):
        raise ValueError('F5 speed需在0.6至1.5。')
    if not (math.isfinite(cfg) and 0 <= cfg <= 5):
        raise ValueError('F5 cfg_strength需在0至5。')
    if steps not in {16, 32, 64}:
        raise ValueError('F5 nfe_step需選16、32或64。')
    return {'speed': speed, 'cfg_strength': cfg, 'nfe_step': steps}
opts = f5_settings({'f5_generation': {'speed': '1.1', 'nfe_step': 32.9}})
assert opts == {'speed':1.1,'cfg_strength':2.0,'nfe_step':32}
print(opts)

## 20　用物件封裝 F5 模型與共用狀態

**來源：** 影片版第 2 格內嵌 video_engine.py　L211–L218

self 代表這個實例。轉接器不是另一個模型，而是讓後面的程式可以用相同的 model.generate 和 model.sr 介面。初始化與逐段推論分開，避免每一句都重新載入權重。

```python
class F5Adapter:
    # Normalise the F5 result to the original V2 Chatterbox-style audio interface.
    def __init__(self, reference_text, options):
        from f5_tts.api import F5TTS
        self.model = F5TTS(model='F5TTS_v1_Base', device='cuda')
        self.sr = int(self.model.target_sample_rate)
        self.reference_text = reference_text
        self.options = options
```

**逐行讀法**

- **L211**　宣告轉接類別 F5Adapter。
- **L212**　註解交代設計目的：對齊既有音訊介面。
- **L213**　__init__ 在建立實例時接收參考逐字稿和已驗證的選項。
- **L214**　在初始化時才載入 F5 的 API。
- **L215**　建立 F5TTS_v1_Base 模型，指定 CUDA。
- **L216**　將模型目標取樣率轉成整數並公開為 self.sr。
- **L217**　保存這次任務共用的參考逐字稿。
- **L218**　保存這次任務共用的有效 F5 選項。

**追蹤一次執行：** 建立一次 adapter 後，即使有兩張圖，也沿用同一個 model 與 reference_text；每張圖變的是 gen_text。

## 21　把共同 generate 呼叫轉接為 F5 infer

**來源：** 影片版第 2 格內嵌 video_engine.py　L220–L226

**unused_chatterbox_options 是接收參數，而 **self.options 是展開參數。兩者方向相反。呼叫端雖然也傳 **generation，這個 F5 方法實際只把建構時保存的 options 交給 infer。

```python
def generate(self, text, language_id, audio_prompt_path, **unused_chatterbox_options):
    import torch
    wav, sr, _ = self.model.infer(ref_file=str(audio_prompt_path),
        ref_text=self.reference_text, gen_text=text, seed=42, **self.options)
    if int(sr) != self.sr:
        raise RuntimeError('F5輸出取樣率不一致。')
    return torch.as_tensor(wav, dtype=torch.float32).reshape(1, -1)
```

**逐行讀法**

- **L220**　保留共同介面；language_id 與多餘的具名參數在此方法中不參與 infer 呼叫。
- **L221**　匯入張量工具，以便統一回傳型別。
- **L222**　呼叫 F5 infer，傳遞參考音路徑；拆包三個結果，底線表示此處不再使用第三項。
- **L223**　分開傳參考逐字稿與欲合成文字，固定 seed=42，展開 self.options。
- **L224**　比較實際回傳取樣率和模型宣告的取樣率。
- **L225**　不一致就中止，不以錯誤速度播放音訊。
- **L226**　轉成 float32 張量，reshape(1,-1) 形成一列、長度自動推算的波形。

**追蹤一次執行：** 若測試模型回傳三個樣本且取樣率一致，後段得到形狀 (1,3)。此介面測試可以驗證參數映射，不能證明音色、口音或對齊品質。

**動手試驗**　只測轉接器的參數邊界：使用假 infer 和最小張量替身，並在離開 with 後還原模組。此練習不初始化模型。

預期：`參數映射與 reshape 呼叫正確`

In [ ]:
class F5Adapter:
    # Normalise the F5 result to the original V2 Chatterbox-style audio interface.
    def __init__(self, reference_text, options):
        from f5_tts.api import F5TTS
        self.model = F5TTS(model='F5TTS_v1_Base', device='cuda')
        self.sr = int(self.model.target_sample_rate)
        self.reference_text = reference_text
        self.options = options

    def generate(self, text, language_id, audio_prompt_path, **unused_chatterbox_options):
        import torch
        wav, sr, _ = self.model.infer(ref_file=str(audio_prompt_path),
            ref_text=self.reference_text, gen_text=text, seed=42, **self.options)
        if int(sr) != self.sr:
            raise RuntimeError('F5輸出取樣率不一致。')
        return torch.as_tensor(wav, dtype=torch.float32).reshape(1, -1)
from types import SimpleNamespace
from unittest.mock import Mock, patch
import sys
adapter = object.__new__(F5Adapter)
adapter.sr, adapter.reference_text = 24000, '大家好'
adapter.options = {'speed':1.1, 'cfg_strength':2.0, 'nfe_step':32}
adapter.model = Mock()
adapter.model.infer.return_value = ([0.1, 0.2], 24000, None)
tensor = Mock()
fake_torch = SimpleNamespace(float32='float32', as_tensor=Mock(return_value=tensor))
with patch.dict(sys.modules, {'torch':fake_torch}):
    adapter.generate('歡迎來到圖書館', 'zh', 'ref.wav', temperature=0.8)
adapter.model.infer.assert_called_once_with(ref_file='ref.wav', ref_text='大家好',
    gen_text='歡迎來到圖書館', seed=42, **adapter.options)
fake_torch.as_tensor.assert_called_once_with([0.1, 0.2], dtype='float32')
tensor.reshape.assert_called_once_with(1, -1)
print('參數映射與 reshape 呼叫正確')

## 22　在迴圈外選定模型

**來源：** 影片版第 2 格內嵌 video_engine.py　L251–L260

generate_story 一開始按引擎只取 f5_settings 或 generation_settings。此處在 for language 之前建立 model，所以一次生成工作只建一次；重新按生成會啟動新 worker。

```python
if not torch.cuda.is_available():
    raise RuntimeError('請使用 Colab T4 GPU 執行階段。')
print('載入語音模型；所有場景共用這一次載入。', flush=True)
if engine_name == 'f5':
    model = F5Adapter(request['ref_text'].strip(), generation)
else:
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS
    model = ChatterboxMultilingualTTS.from_pretrained(device='cuda')
torch.manual_seed(42)
results, errors = [], []
```

**逐行讀法**

- **L251**　檢查執行這個 worker 的 CUDA 是否可用。
- **L252**　沒有 GPU 則停止，不會自動改走 CPU。
- **L253**　輸出進度並立即刷新日誌。
- **L254**　依先前已驗證的引擎名稱分支。
- **L255**　F5 收到去除首尾空白的參考逐字稿與該引擎的有效設定。
- **L256**　另一個合法分支是 Chatterbox。
- **L257**　只有走這個分支才匯入它的模型類別。
- **L258**　將已有權重載入 CUDA。
- **L259**　設定 PyTorch 隨機種子。
- **L260**　建立兩個獨立清單，分別收集成功和錯誤。

**追蹤一次執行：** 兩場景 × 兩語言不等於載入四次。seed 固定也不等於跨環境音訊完全一致；F5 方法另在每次 infer 明確傳 seed。

## 23　分開字幕文字與模型要念的文字

**來源：** 影片版第 2 格內嵌 video_engine.py　L267–L274

三層迴圈依序是語言、圖片、文字分段。text 留給字幕，spoken 給模型。若直接以 spoken 覆蓋 text，燒錄字幕可能不再保留校閱過的繁體字。

```python
for index, scene in enumerate(scenes):
    print(f'{language}：生成場景 {index + 1}/{len(scenes)}', flush=True)
    text = scene['texts'][language].strip()
    spoken = OpenCC('t2s').convert(text) if code == 'zh' else text
    parts = []
    for part in chunks(spoken):
        with torch.inference_mode():
            wav = model.generate(part, language_id=code, audio_prompt_path=str(reference), **generation)
```

**逐行讀法**

- **L267**　依序走訪此語言的各場景，索引從 0 開始。
- **L268**　用 index+1 顯示人閱讀的場景編號。
- **L269**　從校閱後字典取當前語言文字，去除首尾空白。
- **L270**　只有 zh 將繁體轉為簡體供語音模型使用，不改原 text。
- **L271**　每個新場景重設音訊分段清單。
- **L272**　將 spoken 分成較短的輸入。
- **L273**　停用梯度追蹤，僅做推論。
- **L274**　兩個引擎共用 generate 呼叫；傳語言、聲音路徑與有效設定。

**追蹤一次執行：** 校閱文字「歡迎來到圖書館。」進入中文語音分支後，模型可讀到「欢迎来到图书馆。」；字幕仍應保存原來的字。

## 24　把多段波形寫成一個場景 WAV

**來源：** 影片版第 2 格內嵌 video_engine.py　L275–L284

取樣率的單位是樣本／秒，不能把它當成播放速度百分比。np.concatenate 是依序接長，不是把不同句子的音量加在一起。這段只檢查數值，沒有判定人聲是否清晰。

```python
    values = wav.detach().cpu().numpy().reshape(-1)
    if not values.size or not np.isfinite(values).all():
        raise RuntimeError('模型輸出無效音訊。')
    if parts:
        parts.append(np.zeros(int(model.sr * 0.2), dtype=np.float32))
    parts.append(values)
audio = folder / f'scene_{index + 1:02d}.wav'
sf.write(audio, np.concatenate(parts), model.sr, subtype='PCM_16')
audios.append(audio)
texts.append(text)
```

**逐行讀法**

- **L275**　解除梯度關聯、移到 CPU、轉 NumPy，再攤平成一維。
- **L276**　空陣列或包含無限大、NaN 都不接受。
- **L277**　立即停止該段以免寫出壞音訊。
- **L278**　若不是第一段，就在段落之間插入停頓。
- **L279**　以 model.sr × 0.2 計算靜音樣本數。
- **L280**　將本段真正波形加在清單尾端。
- **L281**　依從 1 開始的場景編號建立雙位數檔名。
- **L282**　接起各陣列，以模型取樣率寫 16 位元 PCM WAV。
- **L283**　將音檔路徑收進該語言清單。
- **L284**　另存原 text，供字幕使用。

**追蹤一次執行：** 以 24 kHz 為例，兩個各一秒的段落中間插入 4800 個零，結果為 52800 個樣本，即 2.2 秒。實際值以 model.sr 為準。

## 25　統一圖片方向與透明背景

**來源：** 影片版第 2 格內嵌 video_engine.py　L81–L88

像素數和檔案位元組數限制的是不同成本。此步驟保留原圖比例；輸出影片尺寸在 render_language 的 scale 與 pad 才決定。

```python
with Image.open(source) as image:
    if image.width * image.height > 25_000_000:
        raise ValueError("圖片不得超過 2500 萬像素，請先縮圖。")
    image = ImageOps.exif_transpose(image).convert('RGBA')
    image.thumbnail((2560, 2560))
    canvas = Image.new('RGBA', image.size, 'white')
    canvas.alpha_composite(image)
    canvas.convert('RGB').save(destination, 'PNG')
```

**逐行讀法**

- **L081**　with 開啟圖片，離開區塊時釋放檔案資源。
- **L082**　用寬乘高確認解碼像素數。
- **L083**　超過 2500 萬像素即停止。
- **L084**　套用相機 EXIF 方向，轉成含透明度的 RGBA。
- **L085**　保持比例在 2560×2560 以內縮圖；thumbnail 直接修改物件。
- **L086**　建立與圖片等大的白色畫布。
- **L087**　按透明度將圖片合成到白底。
- **L088**　轉為 RGB，再以 PNG 寫入目的地。

**追蹤一次執行：** 一張 4000×2000 圖會縮至 2560×1280。即使選了直式影片，這一步也不會把圖強拉成 720×1280。

## 26　將旁白秒數向上對齊影格

**來源：** 影片版第 2 格內嵌 video_engine.py　L121–L124

進入迴圈前已確認三個清單等長，避免 zip 靜默截去較長清單的尾端。wav_duration 用音訊 frame 數除以音訊 frame rate，和影片 FPS 的單位要分清楚。

```python
timeline, subtitles, cursor = [], [], 0.0
for index, (image, audio, text) in enumerate(zip(images, audios, texts)):
    duration = math.ceil(wav_duration(audio) * FPS) / FPS
    segment = folder / f'segment_{index:03d}.mp4'
```

**逐行讀法**

- **L121**　為此語言重新建立時間軸、字幕清單和起點零秒。
- **L122**　zip 將同場景的圖、音、字配對；enumerate 另外給出索引。
- **L123**　以 WAV 的秒數乘 FPS，向上取整再除以 FPS，得到可用完整影格表達的秒數。
- **L124**　以零起算的三位數編號建立片段路徑。

**追蹤一次執行：** 旁白 2.01 秒、影片 24 FPS：ceil(48.24)=49 格，場景約 2.041667 秒。下一場的字幕起點也要使用這個對齊後的值。

**動手試驗**　改動旁白秒數，觀察向上取整與一般四捨五入的差異。

預期：`49 2.041667`

In [ ]:
import math
seconds, fps = 2.01, 24
frames = math.ceil(seconds * fps)
assert frames == 49
print(frames, round(frames/fps, 6))

## 27　將靜態圖和旁白編成一個片段

**來源：** 影片版第 2 格內嵌 video_engine.py　L125–L130

清單每個元素都是一個命令列參數。f 字串先填入尺寸；(ow-iw)/2 則是之後由 FFmpeg 評估的置中運算式。輸出音軌轉為 48 kHz 不代表 TTS 模型本身以 48 kHz 推論。

```python
run_ffmpeg(['-loop', '1', '-framerate', str(FPS), '-i', str(image), '-i', str(audio),
    '-map', '0:v:0', '-map', '1:a:0', '-vf',
    f'scale={width}:{height}:force_original_aspect_ratio=decrease,pad={width}:{height}:(ow-iw)/2:(oh-ih)/2:color=black,setsar=1',
    '-af', 'apad', '-t', f'{duration:.8f}', '-r', str(FPS),
    '-c:v', 'libx264', '-preset', 'fast', '-crf', '22', '-pix_fmt', 'yuv420p',
    '-c:a', 'aac', '-ar', '48000', '-ac', '2', '-b:a', '128k', str(segment)])
```

**逐行讀法**

- **L125**　將圖片循環為連續影格，設定輸入幀率，再讀圖片與旁白兩個輸入。
- **L126**　選第一輸入的第一個影像串流、第二輸入的第一個音訊串流，接著設定影像濾鏡。
- **L127**　先等比縮小，再置中補黑邊；setsar=1 設正方形像素。
- **L128**　apad 以靜音補尾端；-t 限制對齊後總秒數；-r 固定輸出幀率。
- **L129**　指定 H.264、編碼速度、品質參數與 yuv420p 像素格式。
- **L130**　將音訊編成 48 kHz、雙聲道 AAC，寫入此場景片段。

**追蹤一次執行：** 橫式畫布遇到直式圖片時，圖片保持比例，左右出現黑邊。只有這一步不會讓照片中的人物或物件產生動作。

## 28　從同一個 cursor 產生 SRT 與時間軸

**來源：** 影片版第 2 格內嵌 video_engine.py　L131–L135

SRT 時間由 srt_time 先換成整數毫秒，再使用 divmod 分離時、分、秒、毫秒。這是由既有腳本導出的場景級字幕，不是 ASR 辨識或逐字對齊。

```python
end = cursor + duration
subtitle_text = '\n'.join(line.strip() for line in text.splitlines() if line.strip())
subtitles.append(f'{index + 1}\n{srt_time(cursor)} --> {srt_time(end)}\n{subtitle_text}\n')
timeline.append({'scene': index + 1, 'start': cursor, 'end': end, 'text': text})
cursor = end
```

**逐行讀法**

- **L131**　場景結束時間等於前一場終點加上本場的影格對齊長度。
- **L132**　逐行去頭尾空白並略過空行，再以換行接回。
- **L133**　以一開始的字幕序號、起訖時間與字幕文字組成一筆 SRT。
- **L134**　保存同一筆場景的序號、秒數與原文字，供 JSON 輸出。
- **L135**　將本場終點接到下一場起點。

**追蹤一次執行：** ZIP 驗證樣本是 0–2 秒和 2–5 秒，因此第二筆從 00:00:02,000 起。正式旁白長度不同，不能照抄這兩組時間。

**動手試驗**　讀取教材封存的時間軸資料，套用原始的 SRT 格式函式。

預期：`00:00:02,000 --> 00:00:05,000`

In [ ]:
def srt_time(seconds):
    milliseconds = round(seconds * 1000)
    hours, milliseconds = divmod(milliseconds, 3600000)
    minutes, milliseconds = divmod(milliseconds, 60000)
    seconds, milliseconds = divmod(milliseconds, 1000)
    return f'{hours:02}:{minutes:02}:{seconds:02},{milliseconds:03}'
timeline = [{'scene': 1, 'start': 0.0, 'end': 2.0, 'text': 'TEST TONE ONLY 測試音，非AI人聲。'}, {'scene': 2, 'start': 2.0, 'end': 5.0, 'text': 'Subtitle pipeline verification. 字幕流程測試。'}]
assert timeline[0]['end'] == timeline[1]['start'] == 2.0
assert timeline[-1]['end'] == 5.0
print(srt_time(timeline[1]['start']), '-->', srt_time(timeline[1]['end']))

## 29　先合成乾淨影片並寫出外掛字幕

**來源：** 影片版第 2 格內嵌 video_engine.py　L140–L145

燒錄開關是在這段之後才判斷。因此無論開或關，SRT 和 timeline.json 都會建立。此 FFmpeg 命令沒有映射字幕串流，所以有外掛 SRT 不等於 MP4 內有可切換的軟字幕軌。

```python
target = folder / 'video.mp4'
run_ffmpeg(['-f', 'concat', '-safe', '1', '-i', 'concat.txt', '-t', f'{cursor:.8f}',
    '-c:v', 'libx264', '-preset', 'fast', '-crf', '22', '-pix_fmt', 'yuv420p',
    '-c:a', 'aac', '-af', 'aresample=async=1:first_pts=0', '-movflags', '+faststart', 'video.mp4'], cwd=folder)
(folder / 'subtitles.srt').write_text('\n'.join(subtitles), encoding='utf-8')
(folder / 'timeline.json').write_text(json.dumps(timeline, ensure_ascii=False, indent=2), encoding='utf-8')
```

**逐行讀法**

- **L140**　先定義乾淨成片的初始路徑 video.mp4。
- **L141**　concat 從前面建立的清單讀片段，總長度用最後的 cursor。
- **L142**　指定輸出影像的 H.264 編碼設定。
- **L143**　對音訊重新取樣以調整時間戳，將 MP4 索引前移；在 folder 中執行。
- **L144**　將所有字幕項目用換行組成 UTF-8 SRT。
- **L145**　將時間軸寫為含中文且有縮排的 JSON。

**追蹤一次執行：** 關閉字幕時，video.mp4 是乾淨版。某些播放器會自動讀取旁邊的字幕檔；檢查是否燒錄時要關閉外掛字幕再看畫面。

## 30　燒錄成功時保留兩種影片

**來源：** 影片版第 2 格內嵌 video_engine.py　L149–L157

Path.replace 改名後，兩個變數仍分別表示各自的路徑。燒錄改了像素，影像不可用 stream copy；聲音未變，故 -c:a copy 合理。中文字型和 FFmpeg subtitles 濾鏡都屬媒體環境依賴。

```python
clean = None
if burn_subtitles:
    clean = folder / 'video_no_subtitles.mp4'
    target.replace(clean)
    try:
        run_ffmpeg(['-i', clean.name, '-vf',
            "subtitles=subtitles.srt:force_style='FontName=Noto Sans CJK TC,FontSize=18,Outline=2,Shadow=0,Alignment=2,MarginV=24'",
            '-c:v', 'libx264', '-preset', 'fast', '-crf', '22', '-c:a', 'copy',
            '-movflags', '+faststart', target.name], cwd=folder)
```

**逐行讀法**

- **L149**　先用 None 表示沒有另一份乾淨版路徑。
- **L150**　只有 burn_subtitles 為真才執行此分支。
- **L151**　指定乾淨備份的檔名。
- **L152**　將已生成的 video.mp4 移到 clean，而非複製一份。
- **L153**　嘗試產生新的 video.mp4。
- **L154**　讀取乾淨影片並使用影像濾鏡。
- **L155**　subtitles 濾鏡讀取 SRT，指定中文字型、描邊、底部置中與垂直邊距。
- **L156**　影像重新編碼，音軌直接複製。
- **L157**　把帶字幕的新影片寫回 target.name，工作目錄為此語言的資料夾。

**追蹤一次執行：** 開啟後 video.mp4 的字無法關閉；要無字畫面須播放 video_no_subtitles.mp4。這個開關選的是產製策略，而非播放器按鈕。

## 31　燒錄失敗時還原檔案並回報

**來源：** 影片版第 2 格內嵌 video_engine.py　L158–L164

raise ... from exc 保留原始錯誤鏈。失敗分支在 return 之前結束，因此磁碟上雖有可播放的 video.mp4，上層仍會把此語言記為失敗，而非放入成功清單。

```python
    except Exception as exc:
        target.unlink(missing_ok=True)
        clean.replace(target)
        raise RuntimeError('字幕燒錄失敗；無字幕影片已保留為video.mp4。請取消字幕勾選再生成。') from exc
return {'video': str(target), 'duration': cursor, 'subtitles': str(folder / 'subtitles.srt'),
        'subtitles_burned': bool(burn_subtitles),
        'clean_video': str(clean) if clean else str(target)}
```

**逐行讀法**

- **L158**　攔截字幕濾鏡或編碼過程所拋出的例外。
- **L159**　刪除可能只寫到一半的目標；不存在也不報錯。
- **L160**　把完好的無字幕檔改回 video.mp4。
- **L161**　拋出帶有原始原因的 RuntimeError；不冒充已燒錄成功。
- **L162**　成功走到這裡才回傳影片、長度與字幕檔位置。
- **L163**　回傳本次的燒錄狀態。
- **L164**　若有另存乾淨版就回傳其路徑，否則回傳主要影片路徑。

**追蹤一次執行：** 除錯時若看到已還原的 MP4 但下載包沒有該語言，兩者並不矛盾：保全檔案與宣告任務成功是不同責任。

**動手試驗**　使用暫存目錄與文字標記模擬失敗檔，不產生或刪除任何真實影片。

預期：`乾淨內容已還原為 video.mp4`

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as tmp:
    target, clean = Path(tmp)/'video.mp4', Path(tmp)/'video_no_subtitles.mp4'
    target.write_bytes(b'CLEAN')
    target.replace(clean)
    try:
        target.write_bytes(b'PARTIAL')
        raise RuntimeError('模擬編碼失敗')
    except RuntimeError:
        target.unlink(missing_ok=True)
        clean.replace(target)
    assert target.read_bytes() == b'CLEAN' and not clean.exists()
    print('乾淨內容已還原為', target.name)

## 32　區分可以繼續的失敗與 GPU 記憶體不足

**來源：** 影片版第 2 格內嵌 video_engine.py　L285–L292

此處的 **result 是字典展開，並非函式參數傳遞。處理範圍取決於 try 放在哪裡：模型初始化若在語言迴圈前就失敗，這段無法把它改成部分成功。

```python
    print(f'{language}：合成影片…', flush=True)
    result = render_language(images, audios, texts, folder, size, request.get('burn_subtitles', False))
    results.append({'language': language, **result})
except torch.cuda.OutOfMemoryError:
    raise RuntimeError('GPU 記憶體不足，請重新啟動執行階段並減少文字。')
except Exception as exc:
    errors.append({'language': language, 'error': str(exc)})
    print(f'{language} 未完成：{exc}', flush=True)
```

**逐行讀法**

- **L285**　向介面顯示此語言開始合成影片。
- **L286**　將所有場景資料與真正的字幕布林值交給 render_language。
- **L287**　只有順利回傳後才把語言及結果欄位合併到成功清單。
- **L288**　優先攔截 GPU 記憶體不足，避免被一般 except 吞下。
- **L289**　改報容易理解的錯誤並向外拋；不繼續下一語言。
- **L290**　其他本語言內的例外交給一般分支。
- **L291**　將語言與錯誤字串加入 errors。
- **L292**　輸出進度後可回到外層迴圈處理下一語言。

**追蹤一次執行：** 兩個語言中，第一個燒錄失敗、第二個成功，則一般情況下 results 與 errors 各一筆；CUDA OOM 則中止整個工作。

## 33　只封裝成功語言的成果

**來源：** 影片版第 2 格內嵌 video_engine.py　L298–L308

前面的 report 已記錄 tts_engine、generation、seed 和 subtitles_burned。字幕旗標是請求設定，判斷個別語言成功仍須看 results/errors。finally 另清除工作區的 reference.wav，不會刪掉原始上傳檔。

```python
(job / 'manifest.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
archive = job / 'multilingual_videos.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(job / 'manifest.json', 'manifest.json')
    for result in results:
        folder = Path(result['video']).parent
        for file in folder.iterdir():
            if file.suffix in {'.mp4', '.wav', '.srt', '.json'}:
                bundle.write(file, file.relative_to(job))
report['zip'] = str(archive)
return report
```

**逐行讀法**

- **L298**　先把 report 寫成 manifest；此刻還沒有最後的 zip 路徑欄位。
- **L299**　建立本次任務的壓縮檔路徑。
- **L300**　用 with 建立 ZIP，離開區塊時完成索引並關檔。
- **L301**　將 manifest 放在 ZIP 根目錄。
- **L302**　逐筆走訪成功的 results。
- **L303**　以該影片的上層目錄定位語言資料夾。
- **L304**　遍歷這個語言的檔案。
- **L305**　只選 MP4、WAV、SRT 和 JSON。
- **L306**　用相對於 job 的路徑寫進 ZIP，保留語言子資料夾。
- **L307**　打包完成後才把 ZIP 路徑加入回應物件。
- **L308**　將結果回給 worker 入口以寫 response.json。

**追蹤一次執行：** 成功的英語燒錄版可包含 en/video.mp4 和 en/video_no_subtitles.mp4。不要假設寫入磁碟的 manifest 一定包含程式稍後才新增的 zip 鍵。

## 34　以檔案大小決定 Notebook 預覽方式

**來源：** 影片版第 6 格　L31–L37

Base64 是傳輸表示，不會改變取樣率，也不會修復壞音軌。影片若只在嵌入播放器沒有聲音，先驗證下載檔的音軌和其他播放器輸出，再追 TTS。

```python
if path.stat().st_size > 20 * 1024 * 1024:
    print('影片大於 20 MB，請用第 7 格下載後播放。')
    return
encoded = base64.b64encode(path.read_bytes()).decode('ascii')
display(HTML('<video controls preload="metadata" width="360">'
             '<source src="data:video/mp4;base64,' + encoded + '" type="video/mp4">'
             '</video>'))
```

**逐行讀法**

- **L031**　在把整支影片讀入記憶體前先檢查是否超過 20 MiB。
- **L032**　超過時提示改用下載方式播放。
- **L033**　提早結束預覽函式；不會讓已完成的生成工作失效。
- **L034**　將位元組 Base64 編碼，再轉成可插入 HTML 的 ASCII 文字。
- **L035**　建立帶 controls 的 video 元素，先載入媒體資訊。
- **L036**　用 data URL 提供 MP4 內容，不依賴外部影片伺服器。
- **L037**　閉合 video 元素並顯示 HTML。

**追蹤一次執行：** 大檔只跳過嵌入預覽，仍可在第 7 格下載已完成成果；預覽失敗不等於推論失敗。

## 圖書館雙場景實作

### 一　先把三種文字放到正確位置

請依 practice／錄音稿與逐字稿.txt 錄下自己的聲音，將實際念出的字填入 ref_text。用 scene_01.png、scene_02.png 依序建立兩個場景；雙場景旁白.csv 的繁中原文是每場的來源文字，英文校對示例用於翻譯後比對。這些檔案是操作素材，Notebook 沒有自動匯入 CSV。

### 二　沿同一條資料追蹤中英文

第一張圖的原文是「歡迎來到圖書館。請先看清楚各區的指示。」。繁中和 English 都選取時，每場應有兩份校閱文字，但只有一個參考錄音。試著只改英文校閱文字：預測會改變哪一支影片的旁白及字幕，再用 results 中的語言欄位核對。

### 三　比較開關的兩個結果

固定場景、語言、引擎、參考音和生成參數，先關閉字幕生成，再開啟。關閉：video.mp4 為乾淨影片，SRT 仍在。開啟成功：video.mp4 帶字，同目錄另有 video_no_subtitles.mp4。要只比較燒錄帶來的差異，可直接對照同一次成功工作輸出的兩支影片。

### 四　分清教材已驗證的範圍

ZIP／expected 的四個案例為橫式關閉、橫式開啟、直式開啟、方形開啟。驗證紀錄均為 5 秒；片中是兩段測試音而非 AI 人聲。這些檔案可查畫面尺寸、時間軸與字幕位置；不能用它們評估 F5 音色。驗證紀錄另列 F5 API 映射 mock 通過、GPU_models 未執行。

### 驗收問題

一、F5 在哪個 Python 裡執行？純語音 8B 在 kernel，影片合成在 .f5venv。二、為何沒勾字幕仍有 SRT？它在條件分支之前寫出。三、為何同一資料夾有乾淨 MP4 卻被列在 errors？燒錄失敗時先還原檔案，再向上拋出例外。